# Thống kê dữ liệu thu thập

:::{note}
Bảng và biểu đồ trong chương này sinh trực tiếp khi biên dịch từ hai tệp tóm tắt `report_book/data/m1_summary.json` và `report_book/data/m1_distributions.csv`. Các nhận định định lượng dùng ký hiệu `[...]` cho đến khi lượt chạy trên toàn bộ 20.000 ID hoàn tất.
:::


## 1. Kết quả thu thập tổng thể

Bảng dưới đây tổng hợp kết quả phân giải và tải nguồn trên 20.000 vị trí ID được giao cho bốn thành viên Nhóm 10.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"]
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8

with open("../data/m1_summary.json", "r", encoding="utf-8") as f:
    m1_summary = json.load(f)

scope = m1_summary["dataset_scope"]
outcomes = m1_summary["crawling_outcomes"]

overview_df = pd.DataFrame([
    {"Chỉ số": "Tổng số vị trí ID ứng viên kiểm tra", "Giá trị": f"{scope['total_candidate_positions']:,}"},
    {"Chỉ số": "Số bài báo hợp lệ phát hiện được", "Giá trị": f"{outcomes['valid_publications']:,}"},
    {"Chỉ số": "Tổng số phiên bản công bố cần thu thập", "Giá trị": f"{outcomes['total_versions']:,}"},
    {"Chỉ số": "Số phiên bản tải mã nguồn TeX thành công", "Giá trị": f"{outcomes['successful_tex_crawls']:,}"},
    {"Chỉ số": "Số phiên bản ở trạng thái PDF hoặc Word", "Giá trị": f"{outcomes['pdf_only_records']:,}"},
    {"Chỉ số": "Tỉ lệ thu thập thành công", "Giá trị": f"{outcomes['overall_success_rate_percent']:.2f}%"},
    {"Chỉ số": "Tỉ lệ bao phủ nguồn TeX trên bài báo hợp lệ", "Giá trị": f"{outcomes['tex_coverage_percent']:.2f}%"}
])
overview_df.style.hide(axis="index").set_caption("Bảng 1.1: Tổng hợp kết quả thu thập dữ liệu Giai đoạn 1")


### Nhận xét về kết quả tổng thể

- Trong 20.000 vị trí ID kiểm tra, tỉ lệ vị trí có bài báo tồn tại đạt `[...]%`. Các vị trí trống thuộc về số hiệu chưa cấp phát trong tháng hoặc bài báo bị rút trước khi lưu kho.
- Tỉ lệ các phiên bản có mã nguồn TeX đạt `[...]%`. Các trường hợp PDF hoặc Word chiếm `[...]%` và đều được ghi nhận bằng bản ghi rỗng để giữ đúng mẫu số thống kê.


## 2. Phân bố số lượng phiên bản

Chương trình thu thập phải tải mọi phiên bản của mỗi bài báo, không chỉ lấy bản mới nhất. Đồ thị dưới đây thể hiện số lượng phiên bản của các bài báo trong tập dữ liệu.


In [ ]:
df_dist = pd.read_csv("../data/m1_distributions.csv")

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)
version_counts = df_dist["version_count"].value_counts().sort_index()
bars = ax.bar(version_counts.index, version_counts.values, color="#1f77b4", width=0.55, edgecolor="black", linewidth=0.5)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f"{int(yval)} ({yval/len(df_dist)*100:.1f}%)", ha="center", va="bottom", fontsize=9)

ax.set_title("Hình 1.1: Phân bố số lượng phiên bản trên mỗi bài báo arXiv", fontsize=12, pad=12)
ax.set_xlabel("Số lượng phiên bản công bố", fontsize=10)
ax.set_ylabel("Số lượng bài báo", fontsize=10)
ax.set_xticks(version_counts.index)
ax.set_ylim(0, max(version_counts.values) * 1.15)
plt.tight_layout()
plt.show()


### Nhận xét về phân bố phiên bản

- Phần lớn bài báo chỉ có một phiên bản `v1`, chiếm `[...]%`. Nhóm có từ hai phiên bản trở lên chiếm `[...]%`, và nhóm có từ bốn phiên bản trở lên chiếm `[...]%`.
- Sự xuất hiện của nhiều phiên bản cho thấy bước khử trùng lặp ở Giai đoạn 2 là cần thiết vì nhiều câu và trích dẫn được giữ nguyên qua các lần cập nhật.


## 3. Ảnh hưởng của việc lọc tệp ảnh

Tệp ảnh trong gói TeX chiếm phần lớn dung lượng nhưng không phục vụ việc phân tích cấu trúc văn bản. Bằng cách chỉ giải nén tệp `.tex` và `.bib`, dung lượng lưu trữ giảm đi rõ rệt.


In [ ]:
size_metrics = m1_summary["size_metrics"]
avg_before_mb = size_metrics["avg_size_before_figure_removal_bytes"] / (1024 * 1024)
avg_after_mb = size_metrics["avg_size_after_figure_removal_bytes"] / (1024 * 1024)
reduction_pct = size_metrics["reduction_percentage"]

fig, ax = plt.subplots(figsize=(7, 4.5), dpi=120)
categories = ["Trước khi lọc", "Sau khi lọc"]
values = [avg_before_mb, avg_after_mb]
colors = ["#d95f02", "#2ca02c"]

bars = ax.bar(categories, values, color=colors, width=0.45, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Dung lượng trung bình mỗi bài báo (MB)", fontsize=10)
ax.set_title("Hình 1.2: So sánh dung lượng trung bình trước và sau khi lọc ảnh", fontsize=12, pad=12)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2.0, val + 0.3, f"{val:.2f} MB", ha="center", va="bottom", fontsize=10)

ax.text(0.5, max(values) * 0.7, f"Giảm: {reduction_pct:.1f}%", fontsize=11, ha="center",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#fff2cc", edgecolor="#d6b656"))

ax.set_ylim(0, max(values) * 1.2)
plt.tight_layout()
plt.show()


### Phân tích dung lượng lưu trữ

- Trước khi lọc, một bài báo có dung lượng trung bình khoảng `[...] MB`. Sau khi lọc bỏ tệp ảnh khi giải nén, dung lượng giảm còn `[...] MB` mỗi bài báo.
- Mức giảm đạt `[...]%`, giúp toàn bộ dữ liệu 20.000 bài có thể lưu trên ổ đĩa của Google Colab mà không đầy đĩa.


## 4. Thống kê dữ liệu tài liệu tham khảo

Bài toán đối sánh ở Giai đoạn 3 yêu cầu trích dẫn có gắn với arXiv ID. Đồ thị dưới đây thể hiện số lượng tài liệu tham khảo trên mỗi bài báo và tỉ lệ có arXiv ID lấy qua API của Semantic Scholar.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=120)

sns.histplot(df_dist["ref_count"], bins=15, kde=True, color="#386cb0", ax=ax1, edgecolor="black", linewidth=0.5)
ax1.set_title("Hình 1.3a: Phân bố số lượng tham chiếu mỗi bài", fontsize=11)
ax1.set_xlabel("Số lượng tài liệu tham khảo", fontsize=9)
ax1.set_ylabel("Số lượng bài báo", fontsize=9)

ref_metrics = m1_summary["reference_metrics"]
labels = ["Có arXiv ID", "Không có arXiv ID"]
sizes = [ref_metrics["references_with_arxiv_id"], ref_metrics["total_crawled_references"] - ref_metrics["references_with_arxiv_id"]]
colors = ["#4daf4a", "#999999"]
ax2.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=140, colors=colors, explode=(0.05, 0), textprops={"fontsize": 9})
ax2.set_title("Hình 1.3b: Tỉ lệ tài liệu tham khảo có arXiv ID", fontsize=11)

plt.tight_layout()
plt.show()


### Nhận xét về dữ liệu tham chiếu

- Trung bình mỗi bài báo chứa `[...]` tài liệu tham khảo, với trung vị dao động từ `[...]` đến `[...]` mục.
- Khoảng `[...]%` tài liệu tham khảo có liên kết với một arXiv ID trên Semantic Scholar. Tập này sẽ chuyển thành các dòng trong bảng `references` để làm tập ứng viên cho Giai đoạn 3.
- Tỉ lệ lấy dữ liệu trích dẫn thành công từ Semantic Scholar đạt `[...]%`. Phần còn lại chưa lấy được do độ trễ lập chỉ mục của bài báo mới hoặc do trích dẫn không đúng định dạng chuẩn.
